# Data Cleaning in Pandas

In [ ]:
import pandas as pd

In [ ]:
url = "https://raw.githubusercontent.com/thibdrv/Toys_and_Models/main/BDD/Tables/"
tables = ["customers", "employees", "offices", "orderdetails", "orders", "productlines", "products"]

data = {t: pd.read_csv(url + f"{t}.csv") for t in tables}

df = (
    data["orderdetails"]
    .merge(data["orders"], on="orderNumber", how="left")
    .merge(data["products"], on="productCode", how="left")
    .merge(data["productlines"], on="productLine", how="left")
    .merge(data["customers"], on="customerNumber", how="left")
    .merge(data["employees"], left_on="salesRepEmployeeNumber", right_on="employeeNumber", how="left")
    .merge(data["offices"], on="officeCode", how="left")
)

# Payments is omitted to prevent duplicates during the merge
# It can be analyzed separately

df.head()

,orderNumber,productCode,quantityOrdered,priceEach,orderLineNumber,orderDate,requiredDate,shippedDate,status,comments,...,reportsTo,jobTitle,city_y,phone_y,addressLine1_y,addressLine2_y,state_y,country_y,postalCode_y,territory
0,10100,S18_1749,30,136.00,3,2018-01-06,2018-01-13,2018-01-10,Shipped,NaN,...,1143.0,Sales Rep,Boston,+1 215 837 0825,1550 Court Place,Suite 102,MA,USA,02107,NaN
1,10100,S18_2248,50,55.09,2,2018-01-06,2018-01-13,2018-01-10,Shipped,NaN,...,1143.0,Sales Rep,Boston,+1 215 837 0825,1550 Court Place,Suite 102,MA,USA,02107,NaN
2,10100,S18_4409,22,75.46,4,2018-01-06,2018-01-13,2018-01-10,Shipped,NaN,...,1143.0,Sales Rep,Boston,+1 215 837 0825,1550 Court Place,Suite 102,MA,USA,02107,NaN
3,10100,S24_3969,49,35.29,1,2018-01-06,2018-01-13,2018-01-10,Shipped,NaN,...,1143.0,Sales Rep,Boston,+1 215 837 0825,1550 Court Place,Suite 102,MA,USA,02107,NaN
4,10101,S18_2325,25,108.06,4,2018-01-09,2018-01-18,2018-01-11,Shipped,Check on availability.,...,1102.0,Sales Rep,London,+44 20 7877 2041,25 Old Broad Street,Level 7,NaN,UK,EC2N 1HN,EMEA


With this merge :

- Y = office
- X = customer

Renaming for clarity :

In [ ]:
df = df.rename(columns=lambda c: c.replace('_x', '_customer').replace('_y', '_office'))

**Global shape**

In [ ]:
df.shape

(2649, 50)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2649 entries, 0 to 2648
Data columns (total 50 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   orderNumber             2649 non-null   int64  
 1   productCode             2649 non-null   object 
 2   quantityOrdered         2649 non-null   int64  
 3   priceEach               2649 non-null   float64
 4   orderLineNumber         2649 non-null   int64  
 5   orderDate               2649 non-null   object 
 6   requiredDate            2649 non-null   object 
 7   shippedDate             2587 non-null   object 
 8   status                  2649 non-null   object 
 9   comments                638 non-null    object 
 10  customerNumber          2649 non-null   int64  
 11  productName             2649 non-null   object 
 12  productLine             2649 non-null   object 
 13  productScale            2649 non-null   object 
 14  productVendor           2649 non-null   

**Duplicated & null**

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.isna().sum().sort_values(ascending=False)

,0
htmlDescription,2649
image,2649
addressLine2_customer,2111
comments,2011
state_office,1545
state_customer,1446
territory,969
addressLine2_office,946
postalCode_customer,128
shippedDate,62


**Drop**

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2649 entries, 0 to 2648
Data columns (total 50 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   orderNumber             2649 non-null   int64  
 1   productCode             2649 non-null   object 
 2   quantityOrdered         2649 non-null   int64  
 3   priceEach               2649 non-null   float64
 4   orderLineNumber         2649 non-null   int64  
 5   orderDate               2649 non-null   object 
 6   requiredDate            2649 non-null   object 
 7   shippedDate             2587 non-null   object 
 8   status                  2649 non-null   object 
 9   comments                638 non-null    object 
 10  customerNumber          2649 non-null   int64  
 11  productName             2649 non-null   object 
 12  productLine             2649 non-null   object 
 13  productScale            2649 non-null   object 
 14  productVendor           2649 non-null   

In [ ]:
df["productLine"]

,productLine
0,Vintage Cars
1,Vintage Cars
2,Vintage Cars
3,Vintage Cars
4,Vintage Cars
...,...
2644,Vintage Cars
2645,Trucks and Buses
2646,Vintage Cars
2647,Trucks and Buses


In [ ]:
df = df.drop(columns=["orderNumber", "productCode", "productDescription", "textDescription", "salesRepEmployeeNumber", "officeCode",
                      "customerNumber", "reportsTo" ,"employeeNumber", "htmlDescription", "image", "orderLineNumber", "extension"])

In [ ]:
df[["state_customer", "state_office"]]

,state_customer,state_office
0,NH,MA
1,NH,MA
2,NH,MA
3,NH,MA
4,NaN,NaN
...,...,...
2644,CA,CA
2645,CA,CA
2646,CA,CA
2647,CA,CA


In [ ]:
df = df.drop(columns=["state_customer", "state_office"])

**Merge**

In [ ]:
df["employeeName"] = (
    df["firstName"].fillna("") + " " + df["lastName"].fillna("")).str.strip()

df = df.drop(columns=["firstName", "lastName"])

df["employeeName"]

,employeeName
0,Steve Patterson
1,Steve Patterson
2,Steve Patterson
3,Steve Patterson
4,Barry Jones
...,...
2644,Leslie Jennings
2645,Leslie Jennings
2646,Leslie Jennings
2647,Leslie Jennings


In [ ]:
df["contactName"] = (
    df["contactFirstName"].fillna("") + " " + df["contactLastName"].fillna("")).str.strip()

df = df.drop(columns=["contactFirstName", "contactLastName"])

df["contactName"]

,contactName
0,Dorothy Young
1,Dorothy Young
2,Dorothy Young
3,Dorothy Young
4,Roland Keitel
...,...
2644,Susan Nelson
2645,Susan Nelson
2646,Susan Nelson
2647,Susan Nelson


In [ ]:
df["address_office"] = (
    df["addressLine1_office"].fillna("") + " " + df["addressLine2_office"].fillna("") + " " +
    df["postalCode_office"].fillna("")
).str.strip()

df = df.drop(columns=["addressLine1_office", "addressLine2_office", "postalCode_office"])

df["address_office"]

,address_office
0,1550 Court Place Suite 102 02107
1,1550 Court Place Suite 102 02107
2,1550 Court Place Suite 102 02107
3,1550 Court Place Suite 102 02107
4,25 Old Broad Street Level 7 EC2N 1HN
...,...
2644,100 Market Street Suite 300 94080
2645,100 Market Street Suite 300 94080
2646,100 Market Street Suite 300 94080
2647,100 Market Street Suite 300 94080


In [ ]:
df["address_customer"] = (
    df["addressLine1_customer"].fillna("") + " " + df["addressLine2_customer"].fillna("") + " " +
    df["postalCode_customer"].fillna("")
).str.strip()

df = df.drop(columns=["addressLine1_customer", "addressLine2_customer", "postalCode_customer"])

df["address_customer"]

,address_customer
0,2304 Long Airport Avenue 62005
1,2304 Long Airport Avenue 62005
2,2304 Long Airport Avenue 62005
3,2304 Long Airport Avenue 62005
4,Lyonerstr. 34 60528
...,...
2644,5677 Strong St. 97562
2645,5677 Strong St. 97562
2646,5677 Strong St. 97562
2647,5677 Strong St. 97562


**Cleaning**

In [ ]:
df["city_customer"].unique()

array(['Nashua', 'Frankfurt', 'NYC', 'Stavern', 'Madrid', 'Kobenhavn',
       'Bergamo', 'Makati City', 'Philadelphia', 'Manchester',
       'San Francisco', 'Luleå', 'San Rafael', 'Paris', 'Charleroi',
       'Singapore', 'Barcelona', 'Salzburg', 'Melbourne', 'Reims',
       'Marseille', 'Nantes', 'Las Vegas', 'London', 'Brickhaven',
       'Glendale', 'Auckland  ', 'Toulouse', 'Chatswood', 'Burlingame',
       'Espoo', 'New Bedford', 'Pasadena', 'North Sydney', 'Brisbane',
       'Oulu', 'South Brisbane', 'Helsinki', 'Milan', 'Los Angeles',
       'Århus', 'Graz', 'Bräcke', 'Montréal', 'Bridgewater',
       'Reggio Emilia', 'Lille', 'Bergen', 'Sevilla', 'Central Hong Kong',
       'Köln', 'Glen Waverly', 'Lyon', 'White Plains', 'New Haven',
       'Burbank', 'Auckland', 'Vancouver', 'Boston', 'Kita-ku',
       'Versailles', 'Dublin', 'Bruxelles', 'San Diego', 'Genève',
       'Cambridge', 'Cowes', 'Newark', 'Wellington', 'Tsawassen',
       'Strasbourg', 'San Jose', 'Liverpool', 'Min

In [ ]:
df["city_customer"] = df["city_customer"].str.strip()

In [ ]:
df["city_office"].unique()

array(['Boston', 'London', 'NYC', 'Paris', 'Tokyo', 'San Francisco',
       'Sydney'], dtype=object)

In [ ]:
df["country_customer"].unique()

array(['USA', 'Germany', 'Norway', 'Spain', 'Denmark', 'Italy',
       'Philippines', 'UK', 'Sweden', 'France', 'Belgium', 'Singapore',
       'Austria', 'Australia', 'New Zealand', 'Finland', 'Canada',
       'Norway  ', 'Hong Kong', 'Japan', 'Ireland', 'Switzerland'],
      dtype=object)

In [ ]:
df["country_customer"] = df["country_customer"].str.strip()

In [ ]:
df["country_office"].unique()

array(['USA', 'UK', 'France', 'Japan', 'Australia'], dtype=object)

In [ ]:
country_to_iso = {
    "USA": "US",
    "Germany": "DE",
    "Norway": "NO",
    "Spain": "ES",
    "Denmark": "DK",
    "Italy": "IT",
    "Philippines": "PH",
    "UK": "GB",
    "Sweden": "SE",
    "France": "FR",
    "Belgium": "BE",
    "Singapore": "SG",
    "Austria": "AT",
    "Australia": "AU",
    "New Zealand": "NZ",
    "Finland": "FI",
    "Canada": "CA",
    "Hong Kong": "HK",
    "Japan": "JP",
    "Ireland": "IE",
    "Switzerland": "CH",
}

In [ ]:
df["phone_customer"].sample(20)

,phone_customer
5,+49 69 66 90 2555
1268,04 499 9555
17,07-98 9555
756,+47 2267 3215
2587,40.67.8555
2266,(93) 203 4555
1761,+65 224 1555
405,6265557265
1631,6175557555
2621,(91) 555 94 44


In [ ]:
# Cleaning phone number

!pip install phonenumbers

In [ ]:
import phonenumbers

def clean_customer(row):
    iso_code = country_to_iso.get(row["country_customer"])
    if not iso_code or pd.isna(row["phone_customer"]):
        return None
    try:
        parsed = phonenumbers.parse(row["phone_customer"], iso_code)
        if phonenumbers.is_valid_number(parsed):
            return phonenumbers.format_number(parsed, phonenumbers.PhoneNumberFormat.INTERNATIONAL)
        else:
            return None
    except phonenumbers.NumberParseException:
        return None

df["phone_customer"] = df.apply(clean_customer, axis=1)

df["phone_customer"]

,phone_customer
0,+1 603-555-8647
1,+1 603-555-8647
2,+1 603-555-8647
3,+1 603-555-8647
4,+49 69 66902555
...,...
2644,+1 415-555-1450
2645,+1 415-555-1450
2646,+1 415-555-1450
2647,+1 415-555-1450


In [ ]:
def clean_office(row):
    iso_code = country_to_iso.get(row["country_office"])
    if not iso_code or pd.isna(row["phone_office"]):
        return None
    try:
        parsed = phonenumbers.parse(row["phone_office"], iso_code)
        if phonenumbers.is_valid_number(parsed):
            return phonenumbers.format_number(parsed, phonenumbers.PhoneNumberFormat.INTERNATIONAL)
        else:
            return None
    except phonenumbers.NumberParseException:
        return None

df["phone_office"] = df.apply(clean_office, axis=1)

df["phone_office"]

,phone_office
0,+1 215-837-0825
1,+1 215-837-0825
2,+1 215-837-0825
3,+1 215-837-0825
4,+44 20 7877 2041
...,...
2644,+1 650-219-4782
2645,+1 650-219-4782
2646,+1 650-219-4782
2647,+1 650-219-4782


**Type**

In [ ]:
df.dtypes

,0
quantityOrdered,int64
priceEach,float64
orderDate,object
requiredDate,object
shippedDate,object
status,object
comments,object
productName,object
productLine,object
productScale,object


In [ ]:
# Error handling

df["requiredDate"] = pd.to_datetime(df["requiredDate"], errors="coerce")

In [ ]:
df["orderDate"] = pd.to_datetime(df["orderDate"])
df["requiredDate"] = pd.to_datetime(df["requiredDate"])
df["shippedDate"] = pd.to_datetime(df["shippedDate"])

**RENAME**

In [ ]:
df = df.rename(columns={"productLine": "category", "email": "employeeEmail",
"comments": "orderComments", "status": "orderStatus"})

In [ ]:
df.head()

,quantityOrdered,priceEach,orderDate,requiredDate,shippedDate,orderStatus,orderComments,productName,category,productScale,...,employeeEmail,jobTitle,city_office,phone_office,country_office,territory,employeeName,contactName,address_office,address_customer
0,30,136.00,2018-01-06,2018-01-13,2018-01-10,Shipped,NaN,1917 Grand Touring Sedan,Vintage Cars,1:18,...,spatterson@classicmodelcars.com,Sales Rep,Boston,+1 215-837-0825,USA,NaN,Steve Patterson,Dorothy Young,1550 Court Place Suite 102 02107,2304 Long Airport Avenue 62005
1,50,55.09,2018-01-06,2018-01-13,2018-01-10,Shipped,NaN,1911 Ford Town Car,Vintage Cars,1:18,...,spatterson@classicmodelcars.com,Sales Rep,Boston,+1 215-837-0825,USA,NaN,Steve Patterson,Dorothy Young,1550 Court Place Suite 102 02107,2304 Long Airport Avenue 62005
2,22,75.46,2018-01-06,2018-01-13,2018-01-10,Shipped,NaN,1932 Alfa Romeo 8C2300 Spider Sport,Vintage Cars,1:18,...,spatterson@classicmodelcars.com,Sales Rep,Boston,+1 215-837-0825,USA,NaN,Steve Patterson,Dorothy Young,1550 Court Place Suite 102 02107,2304 Long Airport Avenue 62005
3,49,35.29,2018-01-06,2018-01-13,2018-01-10,Shipped,NaN,1936 Mercedes Benz 500k Roadster,Vintage Cars,1:24,...,spatterson@classicmodelcars.com,Sales Rep,Boston,+1 215-837-0825,USA,NaN,Steve Patterson,Dorothy Young,1550 Court Place Suite 102 02107,2304 Long Airport Avenue 62005
4,25,108.06,2018-01-09,2018-01-18,2018-01-11,Shipped,Check on availability.,1932 Model A Ford J-Coupe,Vintage Cars,1:18,...,bjones@classicmodelcars.com,Sales Rep,London,+44 20 7877 2041,UK,EMEA,Barry Jones,Roland Keitel,25 Old Broad Street Level 7 EC2N 1HN,Lyonerstr. 34 60528


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2649 entries, 0 to 2648
Data columns (total 29 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   quantityOrdered   2649 non-null   int64         
 1   priceEach         2649 non-null   float64       
 2   orderDate         2649 non-null   datetime64[ns]
 3   requiredDate      2634 non-null   datetime64[ns]
 4   shippedDate       2587 non-null   datetime64[ns]
 5   orderStatus       2649 non-null   object        
 6   orderComments     638 non-null    object        
 7   productName       2649 non-null   object        
 8   category          2649 non-null   object        
 9   productScale      2649 non-null   object        
 10  productVendor     2649 non-null   object        
 11  quantityInStock   2649 non-null   int64         
 12  buyPrice          2649 non-null   float64       
 13  MSRP              2649 non-null   float64       
 14  customerName      2649 n

In [ ]:
df.shape

(2649, 29)

# Download

In [498]:
df.to_csv("Toys_and_Models_Cleaned.csv", index=False)

In [499]:
from google.colab import files

files.download("Toys_and_Models_Cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>